# Análisis Exploratorio de Datos 

_(Metodología CRISP-DM)_

## Introducción  

En este capítulo nos enfocaremos en la **Fase 2 y 3 de CRISP-DM (Comprensión de los Datos y recopilacion de datos)**, realizando un **Análisis Exploratorio de Datos (EDA)** riguroso y estratificado que responde a las particularidades técnicas del proyecto. La estructura del dataset de 1.8 GiB de registros operativos segmentados por camión de los que se tienen los siguientes:
```python
camiones = [(286, 'T-210'), (287, 'T-211'), (288, 'T-212'), (289, 'T-213'),
(290, 'T-214'), (291, 'T-215'), (292, 'T-216'), (293, 'T-217'),
(294, 'T-218'), (295, 'T-219'), (296, 'T-220'), (297, 'T-221'),
(298, 'T-222'), (299, 'T-223'), (300, 'T-224'), (301, 'T-225'),
(302, 'T-230'), (303, 'T-231'), (304, 'T-232'), (305, 'T-233'),
(311, 'T-234'), (312, 'T-235'), (313, 'T-236'), (314, 'T-237'),
(315, 'T-238'), (316, 'T-239'), (317, 'T-240'), (318, 'T-241'),
(319, 'T-242'), (320, 'T-243')]
```

Esta segmentación permite un estudio granular de las variables que impactan el consumo de diésel en cada elemento, garantizando que las transformaciones posteriores y el modelado respeten las diferencias mecánicas y operativas entre equipos. 


## Recopilacion inicial de datos

La elección de los tres conjuntos de datos —eventos, ciclos y sensores— no fue arbitraria. Surgió de un análisis minucioso de la infraestructura tecnológica de MSC, combinado con la necesidad de respetar los procesos ya establecidos por la empresa. A continuación, se detalla el razonamiento detrás de esta decisión, basado en la realidad operativa y los desafíos técnicos identificados.  

## Bases de la Segmentación Fundamentada

MSC gestiona sus datos mediante dos bases clave:  

- **jmineops**: Almacena información cruda en esquemas `dbo` (ej: registros de sensores sin depurar, eventos sin contexto).  
- **mscjmina**: Contiene datos procesados y enriquecidos para reportes, bajo esquemas especializados (ej: `msc_ReportabilidadBase`).  

En las bases de datos se usa un estandar de esquemas como lo son dbo para datos crudos y rep para datos finos de reportes.

Utilizar **mscjmina** y **Msc_Dev** (entorno de desarrollo) como fuentes primarias, ya que encapsulan años de conocimiento operativo de MSC, evitando duplicar esfuerzos. Como punto extra se usa una base de datos especial denominada Msc_Dev para las pruebas y test donde se realizaran mis pruebas para con el proyecto.

## Axioma

Considerot res conjuntos de datos —eventos, ciclos y sensores— desicion no arbitraria. Surgió de un análisis minucioso de la infraestructura tecnológica de MSC, combinado con la necesidad de respetar los procesos ya establecidos por la empresa. A continuación, se detalla el razonamiento detrás de esta decisión, basado en la realidad operativa y los desafíos técnicos identificados. La siguiente diagrama muestra en resumen la forma en la que considero la estructura de los 3 conjuntos de datos:


![image.png](../imgs/Diagrama_influencias.png)

## Los Tres Conjuntos y su Origen Técnico

### Modelo de Tiempos 
- **Fuente**: `[mscjmina].[dbo].[msc_KPI_Ralenti]`

| [mscjmina].[dbo].[msc_KPI_Ralenti] [stored procedure] |
|------------------------------------------------------------------------------------------------|
|Genera un reporte consolidado de eventos y estados operativos de la flota de camiones. Se especializa en identificar cuando los equiposestan en "ralenti".|

- **Contenido**: Estados operativos de los equipos ("Operativo", "En mantenimiento"), tiempos de ralentí y actividad.  
- **Ejemplo de datos**:
  
| id | FechaTurno | created_at | Valor | speed | engine_hours | Anterior_TimeCreate | Tiempo | Turno | Flota | Equipo | Estado | Categoria | Codigo Evento | Evento | Inicio | Fin | Duraciom min | Tiempo_hrs | Ralenti |
|-----|------------|------------|-------|-------|-------------|---------------------|--------|-------|-------|--------|--------|-----------|--------------|---------|--------|-----|-------------|------------|---------|
| 135951869 | 17/3/2025 00:00 | 17/3/2025 07:00 | 982.5 | 9 | 86179,20313 | 17/3/2025 07:00 | 0 | 17-MAR-25 D | CAT 789C | T-210 | Operativo | efectivo | 100 | Producción | 17/3/2025 07:00 | 17/3/2025 07:07 | 7,566666 | 0 | Moviendose |
| 135952047 | 17/3/2025 00:00 | 17/3/2025 07:01 | 1696 | 14 | 86179,46094 | 17/3/2025 07:00 | 1 | 17-MAR-25 D | CAT 789C | T-210 | Operativo | efectivo | 100 | Producción | 17/3/2025 07:00 | 17/3/2025 07:07 | 7,566666 | 0,016666 | Moviendose |

### 2.3. Sensores 
- **Fuente**:

[jmineops].[rep].[custom_RepLibre_hist_Test_Combustible], [rep].[sp_get_RepLibre_Test_Combustible] y El sistema Hexagon Mining en seccion de sensores VIMS.

- **Desafío inicial**: Al analizar las fuentes, nos enfrentamos a un escenario complejo, ya que usan diferentes tablas que se relacionan para dar los datos de sensores, el mas relevantes es shift_sensors que presenta las siguientes caracteristicas:
Datos dispersos: La información crítica (diésel, RPM y velocidad) estaba fragmentada en registros separados, identificados por sensor_set_id distintos:
    Diésel (%) → sensor_set_id=481 (registros cada 30 segundos).
    RPM → sensor_set_id=490 (registros cada 60 segundos).
    Velocidad → Campo independiente en algunos registros.
Inconsistencias graves: Días completos sin datos de RPM pero con diésel, o viceversa. Por ejemplo, el 15% de los registros de RPM tenían valores NULL en horarios críticos.
- **Solución**: Desarrollamos un procedimiento almacenado, que tiene el objetivo de actuar como un "unificador" de estos datos para su posterior analisis, los detalles del mismo estan ubicados en la seccion [1_sensores](./1_sensors.ipynb)

### 2.2. Ciclo Operativo  
- **Fuente**: `[mscjmina].[dbo].[msc_ReportabilidadBase_Ciclos]`.

| [mscjmina].[dbo].[msc_ReportabilidadBase_Ciclos] [stored procedure]|
|------------------------------------------------------------------------------------------------|
|Calcula todos los datos referentes a los ciclos de operaciones de carguio y acarreo de los camiones.|
  
- **Contenido**: Esta tabla detalla operaciones de carguío y acarreo, en la que cada fila representa un ciclo completo de un camión, desde la carga hasta la descarga
- **Ejemplo de dato**:
  
| name          | Pala   | Camion | Frente       | Polygono               | material | tipo Destino | Destino     | Tonelaje Medido | Tonelaje reportado | Zn  | Ag   | Pb  | MPa  | Viaje vacio (s) | Camion espera Pala (s) | Aculatamiento (s) | Cargando (s) | Viaje Lleno (s) | Espera botadero (s) | Retrocediendo (s) | Descarga (s) | Ciclo Acarreo (s) | Tiempo fijo (s) | ciclo total (s) | Hora camion lleno       | Hora camion volteo      | Distancia lleno (m) | Distancia vacio (m) | Distancia equivalente (m) | Vel. vacio (km/h) | Vel. lleno (km/h) | Vel. Promedio (km/h) | easting carga    | northing carga  | easting descarga  | northing descarga | GM             | Productivo  | Assignment         |  
|---------------|--------|--------|--------------|------------------------|----------|--------------|-------------|-----------------|--------------------|-----|------|-----|------|-----------------|-------------------------|--------------------|--------------|-----------------|---------------------|-------------------|--------------|--------------------|-----------------|-----------------|-------------------------|-------------------------|----------------------|---------------------|---------------------------|-------------------|--------------------|-----------------------|-------------------|-----------------|--------------------|--------------------|----------------|------------|--------------------|  
| 13-MAR-25 D   | SV-104 | T-219  | S06-40100A   | S6-4010_49WSS1/SF-WS   | SF-WS    | Botadero     | WD-FASE7RS  | 156,6000061     | 185                | 0,01| 0,16 | 0   | 24,97| 390             | 2                       | 55                | 130          | 1023            | 2                   | 40                | 35           | 1677               | 224             | 1677            | 13/03/2025 06:39:05   | 13/03/2025 07:11:55    | 3161                 | 2860                | 6021                     | 25,2              | 10,8               | 14,4                  | 686879,2119       | 7666498,126     | 685717,4862       | 7666972,335       |                | productivo | Shovel Assignment  |  
| 13-MAR-25 D   | SV-105 | T-210  | S08-41000A   | S8-4100_06MG00/SF-MG   | SF-MG    | Chancador    | PRINCIPAL   | 180             | 180                | 1,2 | 22,15| 0,44| 79,13| 255             | 0                       | 70                | 40           | 275             | 20                  | 0                 | 431           | 1091               | 561             | 1091            | 13/03/2025 06:41:00   | 13/03/2025 07:27:59    | 1376                 | 1143                | 2519                     | 14,4              | 18                 | 14,4                  | 686674,9428       | 7667325,495     |                   |                   | GM01_Alta Ley  | productivo | Shovel Assignment  |  


## Conclusión

Este enfoque se estructura en tres conjuntos principales (eventos, ciclos y sensores), derivados directamente de los sistemas operativos de MSC (jmineops para datos crudos y mscjmina para procesados). Adicionalmente, se incorpora un cuarto dataset externo desde Truck Shop **exclusivo para validación cruzada** que no forma parte del flujo principal, pero actúa como verificador independiente de las métricas de consumo.

La decisión de incluir esta fuente externa surge de una necesidad crítica: asegurar que los cálculos automatizados, por precisos que parezcan, se alineen con registros físicos de suministro documentados.
